In [1]:
from core import EEGModel, CostModel
from preprocessing import list_eegs, load_eeg
from synthetic_profiles import PVProfile

import matplotlib.pyplot as plt

In [27]:
# Terfens
# VS Vomperbach der Gemeinde Terfens - 50 kWp
# Feuerwehr Terfens - 57 kWp
# Heizwerk Terfens - 28 kWp
# Gemeindeamt - 23 kWp FASSADENANALGE
# Bildungszentrum - 35 kWp

pv1 = PVProfile(
    lat=47.322360,
    lon=11.642956,
    peak_power_kw=50+57+28+23+35,
    loss=14.0,
    optimal_angles=False,
    angle=25,
    azimuth=0,
    startyear=2019,
    endyear=2023,
    name="Terfens PV",
)

erzeugung = pv1.profile

c:\Users\paul.toechterle\OneDrive - Energieagentur Tirol GmbH\Desktop\EEG Speicher Tool\synthetic_profiles.py:194: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



In [28]:
import pandas as pd

# drop Feb 29 (leap-day) hours so every year has 8760 hours
mask = ~((erzeugung.index.month == 2) & (erzeugung.index.day == 29))

# Create an "hour-of-year" based on a non-leap reference year (2001) for the masked timestamps.
# Doing this on the masked timestamps avoids trying to map 29-Feb to 2001 (which would be invalid).
ref_dates = pd.DatetimeIndex(erzeugung.index[mask].map(lambda ts: ts.replace(year=2001)))
hoy_ref = (ref_dates.dayofyear - 1) * 24 + ref_dates.hour

# average across years for each hour-of-year (0..8759)
avg_annual_profile = erzeugung[mask].groupby(hoy_ref).mean()
assert len(avg_annual_profile) == 8760

# give it a representative non-leap datetime index for plotting (e.g. year 2001)
rep_index = pd.date_range('2025-01-01', periods=8760, freq='h', tz=erzeugung.index.tz)
avg_annual_profile.index = rep_index

In [29]:
df_terfens = load_eeg(list_eegs()[1])
df_terfens.columns

Index(['bez_ges', 'bez_rest', 'bez_eff', 'lief_ges', 'lief_rest', 'lief_eff'], dtype='object')

In [30]:
from plotly.subplots import make_subplots

import plotly.graph_objects as go

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.04,
    subplot_titles=("PV Erzeugung", "Lieferung gesamt", "Bezug gesamt")
)

fig.add_trace(
    go.Scatter(x=avg_annual_profile.index, y=avg_annual_profile.values, mode="lines", name="PV Synthetic"),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df_terfens.index, y=df_terfens["lief_ges"], mode="lines", name="Lieferung gesamt"),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df_terfens.index, y=df_terfens["bez_ges"], mode="lines", name="Bezug gesamt"),
    row=3, col=1
)

fig.update_xaxes(row=3, col=1, rangeslider_visible=False)
fig.update_layout(height=800, width=1200, template="plotly_white", legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig.show()